In [51]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
import sys
import os
from datetime import datetime

project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.utils.team_info import *
from src.historical_analysis.dataScraper import *

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

In [53]:
import joblib

project_root = Path("/Users/alexgonzalez/Documents/NBA-Prop-Predictor")  # or Path.cwd()
min_path = project_root / "src/models/saved_models/min_quantile_xgb_2026-01-02.joblib"
ppm_path = project_root / "src/models/saved_models/ppm_quantile_xgb_2025-12-31.joblib"

min_bundle = joblib.load(min_path)
min_models = min_bundle["quantile_models"]
min_feature_names = min_bundle["feature_names"]

ppm_bundle = joblib.load(ppm_path)
ppm_models = ppm_bundle["quantile_models"]
ppm_feature_names = ppm_bundle["feature_names"]


#load season stats
pts_df = pd.read_csv('data/processed/training/S26_TRAINING_PPM.csv')
ast_df = pd.read_csv('data/processed/training/S26_TRAINING_APM.csv')
reb_df = pd.read_csv('data/processed/training/S26_TRAINING_RPM.csv')
min_df = pd.read_csv('data/processed/training/S26_TRAINING_MIN.csv')

### Helper Functions

In [54]:
from src.live import *

def get_rate_history(df, player, date, n_games=20, rate_col='PTS_PER_MIN'):
    pdf = df[(df['PLAYER_NAME'] == player) & (df['GAME_DATE'] < date)]
    pdf = pdf.sort_values(by='GAME_DATE', ascending=True)
    rate_history = pdf[rate_col].dropna().tail(n_games)
    return rate_history

def grab_player_last_game(df, features, player, date):
    pdf = df[(df['PLAYER_NAME'] == player) & (df['GAME_DATE'] == date)]
    if pdf.empty:
        return f"No data found for {player} on {date}"
    return pdf[features]

def build_sim_row(player, min_arr, ppm_arr, min_models, ppm_models, rate_history):
    """
    Build a row dict compatible with `run_pts_simulation` in src/live.py.
    Predicts q10/q50/q90 for both MIN and PPM and attaches rate history.
    """
    q10, q50, q90 = 'q_0.10', 'q_0.50', 'q_0.90'

    m10 = float(min_models[q10].predict(min_arr)[0])
    m50 = float(min_models[q50].predict(min_arr)[0])
    m90 = float(min_models[q90].predict(min_arr)[0])

    r10 = float(ppm_models[q10].predict(ppm_arr)[0])
    r50 = float(ppm_models[q50].predict(ppm_arr)[0])
    r90 = float(ppm_models[q90].predict(ppm_arr)[0])

    return {
        'PLAYER_NAME': player,
        'MARKET': 'PTS',
        'MIN_Q10': m10, 'MIN_Q50': m50, 'MIN_Q90': m90,
        'RATE_Q10': r10, 'RATE_Q50': r50, 'RATE_Q90': r90,
        'STAT_Q10': m10 * r10, 'STAT_Q50': m50 * r50, 'STAT_Q90': m90 * r90,
        'RATE_HISTORY': list(rate_history) if rate_history is not None else [],
    }


### Solo lookup

In [55]:
player = 'Shai Gilgeous-Alexander'
date = '2026-04-02'
last_mins = grab_player_last_game(min_df, min_feature_names, player, date)
last_ppm = grab_player_last_game(pts_df, ppm_feature_names, player, date)

min_arr = np.asarray(last_mins, dtype=float).reshape(1, -1)
ppm_arr = np.asarray(last_ppm, dtype=float).reshape(1, -1)

min_pred = min_models['q_0.50'].predict(min_arr)
ppm_pred = ppm_models['q_0.50'].predict(ppm_arr)
ppm_pred_q90 = ppm_models['q_0.90'].predict(ppm_arr)
rate_history = get_rate_history(pts_df, player, '2026-04-02').to_list()

print(f"predicted MIN: {min_pred[0]:.2f}, predicted PPM: {ppm_pred[0]:.2f}, predicted PTS: {min_pred[0] * ppm_pred[0]:.2f}, predicted PTS q90: {min_pred[0] * ppm_pred_q90[0]:.2f}")
print(f"rate history: {rate_history}")
last_ppm

predicted MIN: 34.96, predicted PPM: 0.96, predicted PTS: 33.57, predicted PTS q90: 41.67
rate history: [0.6818827540486789, 0.8094152672465925, 0.9549071618037136, 1.016949152542373, 0.7194244604316546, 1.0661401776900297, 0.8962804361898123, 0.7330827067669172, 0.7714285714285715, 0.8985879332477534, 0.8955223880597014, 0.6018054162487462, 1.1230697239120262, 0.7822685788787485, 1.2572027239392354, 0.7573149741824441, 0.8928571428571428, 0.8479366873940078, 0.8148483476686282, 1.1595394736842106]


,UFGA_PER_MIN_X_OPP_DEF_RATING,CFGA_PER_MIN_X_OPP_DEF_RATING,PPM_SEASON_MEAN,TEAM_USG_RANK_L10,PTS_PER_MIN_X_OPP_PTS_ALLOWED,TS_PCT_X_USG_PCT,FT_PCT_season_avg,PTS_season_avg
147073,45.9,28.35,0.944005,1.0,106.66821,0.2176,0.89,31.62


### Week lookup 

In [78]:
date = '2025-12-25'  # game date for backtest
N_SIMS = 10_000
PROB_THRESHOLD = 0.58
STAKE = 100.0

np.random.seed(42)

backtest_df = pd.read_csv(
    '/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/raw/player_lines/NBA_US_20251225_135017.csv'
)
backtest_df = backtest_df[backtest_df['CATEGORY'] == 'player_points'].copy()

# Pivot so each (bookmaker, player, line) row has both Over + Under odds
lines_df = (
    backtest_df
    .pivot_table(
        index=['BOOKMAKER', 'NAME', 'LINE'],
        columns='OVER/UNDER',
        values='ODDS',
        aggfunc='first',
    )
    .reset_index()
    .rename(columns={'Over': 'odds_over', 'Under': 'odds_under'})
)

def american_to_profit(odds, stake=STAKE):
    odds = float(odds)
    if odds > 0:
        return stake * (odds / 100.0)
    return stake * (100.0 / abs(odds))

# Cache simulations per player so we don't redo them for every bookmaker
sim_cache = {}

def get_sims_for_player(player):
    if player in sim_cache:
        return sim_cache[player]

    pdf = pts_df[(pts_df['PLAYER_NAME'] == player) & (pts_df['GAME_DATE'] == date)]
    if pdf.empty:
        sim_cache[player] = None
        return None

    last_mins = grab_player_last_game(min_df, min_feature_names, player, date)
    last_ppm = grab_player_last_game(pts_df, ppm_feature_names, player, date)
    if not isinstance(last_mins, pd.DataFrame) or not isinstance(last_ppm, pd.DataFrame):
        sim_cache[player] = None
        return None

    min_arr = np.asarray(last_mins, dtype=float).reshape(1, -1)
    ppm_arr = np.asarray(last_ppm, dtype=float).reshape(1, -1)
    rate_history = get_rate_history(pts_df, player, date, n_games=20).tolist()
    sim_row = build_sim_row(player, min_arr, ppm_arr, min_models, ppm_models, rate_history)
    sims = run_pts_simulation(sim_row, n_sims=N_SIMS)

    payload = {
        'sims': sims,
        'sim_row': sim_row,
        'actual_pts': pdf['PTS'].values[0],
        'actual_min': pdf['MIN'].values[0],
        'player_pts_mean': pts_df[pts_df['PLAYER_NAME'] == player]['PTS'].mean(),
    }
    sim_cache[player] = payload
    return payload

res = []
missing_players = set()

for _, row in lines_df.iterrows():
    player = row['NAME']
    book = row['BOOKMAKER']
    line = float(row['LINE'])
    odds_over = row.get('odds_over')
    odds_under = row.get('odds_under')

    payload = get_sims_for_player(player)
    if payload is None:
        missing_players.add(player)
        continue

    sims = payload['sims']
    sim_row = payload['sim_row']
    actual_pts = payload['actual_pts']

    p_over = float(np.mean(sims > line))
    p_under = float(np.mean(sims < line))
    sim_mean = float(np.mean(sims))
    sim_p10, sim_p50, sim_p90 = np.percentile(sims, [10, 50, 90])

    pred = sim_row['STAT_Q50']
    side = 'Over' if p_over >= p_under else 'Under'
    p_side = p_over if side == 'Over' else p_under
    used_odds = odds_over if side == 'Over' else odds_under

    if pd.isna(used_odds):
        continue

    hit = (side == 'Over' and actual_pts > line) or (side == 'Under' and actual_pts < line)
    edge = abs(pred - line)
    recommended_bet = 1 if ((p_side >= PROB_THRESHOLD) and (edge > 2.5)) else 0
    profit = american_to_profit(used_odds) if hit else -STAKE

    res.append({
        'bookmaker': book,
        'player': player,
        'player_pts_mean': payload['player_pts_mean'],
        'pred_points': round(pred, 2),
        'sim_mean': round(sim_mean, 2),
        'sim_p10': round(float(sim_p10), 2),
        'sim_p50': round(float(sim_p50), 2),
        'sim_p90': round(float(sim_p90), 2),
        'side': side,
        'line': line,
        'odds_used': used_odds,
        'odds_over': odds_over,
        'odds_under': odds_under,
        'actual_points': actual_pts,
        'p_over': round(p_over, 3),
        'p_under': round(p_under, 3),
        'p_side': round(p_side, 3),
        'edge': round(edge, 2),
        'hit': int(hit),
        'stake': STAKE,
        'profit': round(profit, 2),
        'recommended_bet': recommended_bet,
    })

res = pd.DataFrame(res)

print('-' * 100)
print(f"Books: {res['bookmaker'].nunique()} | Players: {res['player'].nunique()} | Bets: {len(res)}")
print(f"Hit rate: {round(res['hit'].mean(), 3)}")
print(f"Total staked: ${len(res) * STAKE:.2f} | Profit: ${res['profit'].sum():.2f} "
      f"| ROI: {round(res['profit'].sum() / (len(res) * STAKE) * 100, 2)}%")

rec = res[res['recommended_bet'] == 1]
if len(rec):
    print('-' * 100)
    print(f"Recommended bets (p_side >= {PROB_THRESHOLD}): {len(rec)} | hit rate: {round(rec['hit'].mean(), 3)}")
    print(f"Staked: ${len(rec) * STAKE:.2f} | Profit: ${rec['profit'].sum():.2f} "
          f"| ROI: {round(rec['profit'].sum() / (len(rec) * STAKE) * 100, 2)}%")

if missing_players:
    print('-' * 100)
    print(f"Skipped {len(missing_players)} players with no data on {date}: "
          f"{sorted(missing_players)[:10]}{' ...' if len(missing_players) > 10 else ''}")

res.head(20)

----------------------------------------------------------------------------------------------------
Books: 6 | Players: 76 | Bets: 826
Hit rate: 0.556
Total staked: $82600.00 | Profit: $-2808.15 | ROI: -3.4%
----------------------------------------------------------------------------------------------------
Recommended bets (p_side >= 0.58): 204 | hit rate: 0.686
Staked: $20400.00 | Profit: $1610.92 | ROI: 7.9%
----------------------------------------------------------------------------------------------------
Skipped 1 players with no data on 2025-12-25: ['Jalen Pickett']


,bookmaker,player,player_pts_mean,pred_points,sim_mean,sim_p10,sim_p50,sim_p90,side,line,odds_used,odds_over,odds_under,actual_points,p_over,p_under,p_side,edge,hit,stake,profit,recommended_bet
0,BetMGM,Aaron Wiggins,8.471850,7.15,8.76,3.00,7.26,17.00,Under,7.5,-120,-110,-120,5,0.479,0.520,0.520,0.35,1,100.0,83.33,0
1,BetMGM,Al Horford,9.571749,6.78,7.06,1.85,6.74,12.89,Over,6.5,100,100,-135,14,0.528,0.472,0.528,0.28,1,100.0,100.00,0
2,BetMGM,Alex Caruso,7.266150,5.51,6.38,2.08,5.76,11.99,Under,6.5,-140,105,-140,12,0.409,0.591,0.591,0.99,0,100.0,-100.00,0
3,BetMGM,Alperen Sengun,17.029891,25.99,24.82,13.04,24.88,35.66,Over,21.5,-118,-118,-115,14,0.659,0.341,0.659,4.49,0,100.0,-100.00,1
4,BetMGM,Amen Thompson,14.427273,18.33,18.68,9.62,17.83,28.44,Over,17.5,-115,-115,-118,26,0.519,0.480,0.519,0.83,1,100.0,86.96,0
5,BetMGM,Anthony Davis,23.850467,22.83,24.75,12.75,24.94,37.28,Over,24.5,-110,-110,-118,3,0.520,0.480,0.520,1.67,0,100.0,-100.00,0
6,BetMGM,Anthony Edwards,24.823293,29.06,29.61,15.50,29.02,43.84,Under,30.5,-120,-110,-120,44,0.449,0.551,0.551,1.44,0,100.0,-100.00,0
7,BetMGM,Austin Reaves,15.935657,21.52,24.93,11.32,24.56,39.24,Over,19.5,-105,-105,-125,12,0.649,0.351,0.649,2.02,0,100.0,-100.00,0
8,BetMGM,Brandin Podziemski,11.625000,11.47,13.07,5.95,12.67,20.78,Over,10.5,-118,-118,-110,13,0.654,0.346,0.654,0.97,1,100.0,84.75,0
9,BetMGM,Bruce Brown,9.721277,8.77,9.89,2.18,10.08,16.41,Over,8.5,-105,-105,-125,7,0.628,0.372,0.628,0.27,0,100.0,-100.00,0


In [79]:
def summarize_by_book(df, label):
    if df.empty:
        print(f"No bets for: {label}")
        return None
    out = (
        df.groupby('bookmaker')
          .agg(
              bets=('hit', 'size'),
              hits=('hit', 'sum'),
              hit_rate=('hit', 'mean'),
              staked=('stake', 'sum'),
              profit=('profit', 'sum'),
          )
          .assign(roi_pct=lambda x: (x['profit'] / x['staked'] * 100).round(2))
          .sort_values('roi_pct', ascending=False)
    )
    out['hit_rate'] = out['hit_rate'].round(3)
    out['profit'] = out['profit'].round(2)
    out['staked'] = out['staked'].round(2)
    print(f"\n=== ROI per bookmaker — {label} ===")
    print(out.to_string())
    return out

book_all = summarize_by_book(res, 'all bets')
book_rec = summarize_by_book(res[res['recommended_bet'] == 1],
                             f'recommended bets (p_side >= {PROB_THRESHOLD})')


=== ROI per bookmaker — all bets ===
              bets  hits  hit_rate   staked   profit  roi_pct
bookmaker                                                    
BetOnline.ag    57    32     0.561   5700.0   254.65     4.47
DraftKings      58    32     0.552   5800.0   197.39     3.40
BetMGM          61    33     0.541   6100.0    41.83     0.69
FanDuel         56    30     0.536   5600.0     0.81     0.01
BetRivers      126    66     0.524  12600.0  -574.66    -4.56
Bovada         468   266     0.568  46800.0 -2728.17    -5.83

=== ROI per bookmaker — recommended bets (p_side >= 0.58) ===
              bets  hits  hit_rate   staked   profit  roi_pct
bookmaker                                                    
DraftKings       9     7     0.778    900.0   371.29    41.25
BetOnline.ag     8     6     0.750    800.0   284.68    35.58
Bovada         149   107     0.718  14900.0  1165.80     7.82
BetMGM           9     5     0.556    900.0    16.52     1.84
BetRivers       21    11     0.